# Fase 3: Exploracion y limpieza

Proyecto: evaluacion de salud/riesgo de repositorios de GitHub (GH Archive).

En esta fase:
1. Cargamos las 12 horas crudas descargadas en la Fase 2 (capa **Bronze**).
2. Exploramos los datos: cuantos eventos hay por tipo y por mes (para revisar
   el patron raro que vimos en Fase 2: de marzo 2026 en adelante bajan mucho
   los Issues/PR/Fork/Watch mientras suben los Push).
3. Limpiamos (capa **Silver**): nos quedamos solo con los 5 tipos de evento
   de interes, quitamos bots, quitamos duplicados y filas con campos nulos
   criticos (repo, actor, fecha).
4. Guardamos el resultado limpio en `data/processed/` para usarlo en la
   siguiente fase (features + clustering).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, substring

# Sesion local, sin cluster real (todo corre en esta misma maquina).
# Subimos la memoria del driver porque por defecto Spark solo da 1g,
# y con los 12 archivos juntos (~477MB comprimidos) se queda sin memoria.
spark = (SparkSession.builder
         .master("local[*]")
         .appName("limpieza-gharchive")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/11 21:22:48 WARN Utils: Your hostname, katana, resolves to a loopback address: 127.0.1.1; using 192.168.18.59 instead (on interface enp2s0)
26/07/11 21:22:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/11 21:22:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Capa Bronze: cargar los 12 archivos crudos

Leemos los 12 `.json.gz` de `data/raw/` en un solo DataFrame de Spark.
Cada linea del archivo es un evento en JSON, Spark los infiere solo.

In [2]:
# El comodin *.json.gz toma los 12 archivos de una sola vez
df_bronze = spark.read.json("../data/raw/*.json.gz")

print("total de eventos (bronze, sin filtrar):", df_bronze.count())
df_bronze.printSchema()

total de eventos (bronze, sin filtrar): 1844373
root
 |-- actor: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- display_login: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- org: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- payload: struct (nullable = true)
 |    |-- action: string (nullable = true)
 |    |-- assignee: struct (nullable = true)
 |    |    |-- avatar_url: string (nullable = true)
 |    |    |-- events_url: string (nullable = true)
 |    |    |-- followers_url: string (nullable = true)
 |    |    |-- following_url: strin

## Exploracion: revisar el patron raro por mes

En la Fase 2 notamos que de marzo 2026 en adelante los conteos de
Issues/PR/Fork/Watch bajaban mucho mientras Push subia. Aca lo confirmamos
agrupando por mes y tipo de evento, sobre los 5 tipos de interes (todavia
sin quitar bots, para ver el dato crudo).

In [3]:
EVENTOS_DE_INTERES = ["PushEvent", "IssuesEvent", "PullRequestEvent", "WatchEvent", "ForkEvent"]

df_interes = df_bronze.filter(col("type").isin(EVENTOS_DE_INTERES))

# "created_at" viene como texto tipo 2026-06-16T19:00:00Z -> los primeros
# 7 caracteres son el mes (AAAA-MM)
df_interes = df_interes.withColumn("mes", substring(col("created_at"), 1, 7))

(df_interes
    .groupBy("mes", "type")
    .agg(count("*").alias("cantidad"))
    .orderBy("mes", "type")
    .show(60))

+-------+----------------+--------+
|    mes|            type|cantidad|
+-------+----------------+--------+
|2025-08|       ForkEvent|    1366|
|2025-08|     IssuesEvent|    4018|
|2025-08|PullRequestEvent|   14174|
|2025-08|       PushEvent|   92982|
|2025-08|      WatchEvent|    5393|
|2025-09|       ForkEvent|    1172|
|2025-09|     IssuesEvent|    3742|
|2025-09|PullRequestEvent|   13312|
|2025-09|       PushEvent|  104812|
|2025-09|      WatchEvent|    4633|
|2025-10|       ForkEvent|    1163|
|2025-10|     IssuesEvent|    4638|
|2025-10|PullRequestEvent|   10442|
|2025-10|       PushEvent|   91077|
|2025-10|      WatchEvent|    3712|
|2025-11|       ForkEvent|     577|
|2025-11|     IssuesEvent|    3568|
|2025-11|PullRequestEvent|   12356|
|2025-11|       PushEvent|  103262|
|2025-11|      WatchEvent|    2349|
|2025-12|       ForkEvent|     844|
|2025-12|     IssuesEvent|    3904|
|2025-12|PullRequestEvent|   11204|
|2025-12|       PushEvent|  100909|
|2025-12|      WatchEvent|  

## Capa Silver: limpieza

Pasos, en orden (mostramos cuantas filas quedan despues de cada uno):

1. Quedarnos solo con los 5 tipos de evento de interes (ya lo hicimos arriba).
2. Quitar filas con campos nulos criticos: repo, actor o fecha (asi el
   filtro de bots del siguiente paso no se confunde con actores nulos).
3. Quitar bots (login termina en `-bot`, `[bot]`, o contiene `dependabot`).
4. Quitar eventos duplicados (cada evento de GH Archive trae un `id` unico).

In [4]:
print("1. despues de filtrar tipos de interes:", df_interes.count())

# Paso 2: nulos criticos (repo, actor, fecha)
df_sin_nulos = df_interes.filter(
    col("repo.name").isNotNull()
    & col("actor.login").isNotNull()
    & col("created_at").isNotNull()
)
print("2. despues de quitar nulos criticos:", df_sin_nulos.count())

1. despues de filtrar tipos de interes: 1501990


2. despues de quitar nulos criticos: 1501976


In [5]:
from pyspark.sql.functions import lower

# Paso 3: quitar bots (mismo criterio que usamos en src/ingest.py en Fase 1)
login = lower(col("actor.login"))
es_bot = login.endswith("-bot") | login.endswith("[bot]") | login.contains("dependabot")

df_sin_bots = df_sin_nulos.filter(~es_bot)
print("3. despues de quitar bots:", df_sin_bots.count())

3. despues de quitar bots: 1203097


In [6]:
# Paso 4: quitar duplicados exactos por id de evento (cada evento de
# GH Archive deberia tener un id unico)
df_silver = df_sin_bots.dropDuplicates(["id"])
print("4. despues de quitar duplicados:", df_silver.count())

4. despues de quitar duplicados: 1203093


## Guardar la capa Silver

Guardamos en formato Parquet (formato estandar de Spark, mas rapido de leer
que JSON) para usarlo en la fase de features/clustering.

In [7]:
RUTA_SALIDA = "../data/processed/eventos_silver.parquet"

df_silver.write.mode("overwrite").parquet(RUTA_SALIDA)

# Verificacion rapida: releer y contar
verificacion = spark.read.parquet(RUTA_SALIDA)
print("filas guardadas en", RUTA_SALIDA, ":", verificacion.count())

filas guardadas en ../data/processed/eventos_silver.parquet : 1203093
